In [ ]:
# import TSS data 
import pandas as pd, numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import classification_report, average_precision_score, roc_auc_score

# --- Load data (your paths) ---
pos = pd.read_csv("/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/training_features_values/positive_values/merged_TSS_500bp_20bins.csv")
neg = pd.read_csv("/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/training_features_values/negative_values/ATAC_negatives_2k_features_TSS_only.csv")

pos["label"] = 1
neg["label"] = 0
data = pd.concat([pos, neg], ignore_index=True)

# Build features (drop label + TSS_coord if present)
X_df = data.drop(columns=[c for c in ["label", "TSS_coord"] if c in data.columns], errors="ignore")
X_df = X_df.select_dtypes(include=["number"])
feature_names = X_df.columns.tolist()

print("Total features:", X_df.shape[1])

print(X_df)

In [ ]:
import re

# select biology features 
GROUP_A = {"H2A.Z","H2B.Z","H3K27ac","H3K9ac","H3K18ac","H3K4me3","H2A.Zac","ATAC"}   # bins 01–15
GROUP_B = {"H3","H3K36me3","H3K4me","MNase","H3K27me","H3K4me2","H3R17me2","H3K18me","H3K4me1"}  # bins 07–15

pat = re.compile(r"^(?P<mark>.+?)_TSS_bin(?P<bin>\d+)$")

def keep(col: str) -> bool:
    m = pat.match(col)
    if not m:
        return False
    mark = m.group("mark")
    b = int(m.group("bin"))
    return (mark in GROUP_A and 1 <= b <= 15) or (mark in GROUP_B and 7 <= b <= 15)

# Select and (optionally) sort by mark then bin
selected_cols = [c for c in X_df.columns if keep(c)]
selected_cols = sorted(selected_cols, key=lambda c: (pat.match(c).group("mark"),
                                                     int(pat.match(c).group("bin"))))

X_sel = X_df[selected_cols].copy()
feature_names = selected_cols
print(f"Selected {len(selected_cols)}/{X_df.shape[1]} features")
print(X_sel)

# y from the 'label' column, aligned to X_sel's index
y = data.loc[X_sel.index, "label"].astype(int)

# quick sanity checks
assert len(y) == len(X_sel)
print(y.value_counts())

In [ ]:
# ANNOVA and de-correlate
import numpy as np, pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import f_classif

# --- CONFIG (tune as needed) ---
Q_FDR_CUTOFF = 0.05      # BH-FDR threshold for p-values
ETA2_MIN     = 0.01      # minimum effect size
CORR_THRESH  = 0.95      # max allowed absolute correlation between kept features
TOPK_FALLBACK = 300      # if FDR/eta2 keep too few, fall back to top-K by F

# X_sel: your pre-filtered feature matrix (numeric DataFrame)
# y:     labels (0/1 or multi-class)

# 1) Median-impute for stats
imp = SimpleImputer(strategy="median")
X_imp = pd.DataFrame(imp.fit_transform(X_sel), columns=X_sel.columns, index=X_sel.index)

# 2) ANOVA (per-feature)
F, p = f_classif(X_imp.values, y.values)
F = np.nan_to_num(F, nan=0.0, posinf=0.0, neginf=0.0)
p = np.nan_to_num(p, nan=1.0, posinf=1.0, neginf=1.0)

# 2b) BH-FDR (with safe fallback if statsmodels unavailable)
try:
    from statsmodels.stats.multitest import multipletests
    q = multipletests(p, method="fdr_bh")[1]
except Exception:
    # simple BH implementation
    m = len(p)
    order = np.argsort(p)
    ranked = np.empty_like(order)
    ranked[order] = np.arange(1, m+1)
    q = p * m / ranked
    # enforce monotonicity
    q_sorted = np.minimum.accumulate(q[order][::-1])[::-1]
    q = np.empty_like(q_sorted)
    q[order] = q_sorted

# effect size eta^2
k = y.nunique(); N = len(y)
eta2 = (F * (k - 1)) / (F * (k - 1) + (N - k) + 1e-12)

anova_tbl = pd.DataFrame({
    "feature": X_sel.columns,
    "F": F, "p": p, "q": q, "eta2": eta2
}).sort_values("F", ascending=False).reset_index(drop=True)

# 3) Keep by FDR & effect size (or fallback to top-K)
keep_anova = anova_tbl.query("q < @Q_FDR_CUTOFF and eta2 >= @ETA2_MIN")["feature"].tolist()
if len(keep_anova) == 0:
    keep_anova = anova_tbl.head(min(TOPK_FALLBACK, len(anova_tbl)))["feature"].tolist()

X_anova = X_imp[keep_anova]

# 4) De-correlate greedily by ANOVA rank
#    Traverse features (high→low F) and keep a feature only if it's not highly correlated with any already kept.
corr = X_anova.corr().abs()
ordered_feats = anova_tbl[anova_tbl["feature"].isin(keep_anova)]["feature"].tolist()  # sorted by F desc

kept = []
for f in ordered_feats:
    if not kept:
        kept.append(f)
        continue
    # check correlation with all already kept
    if (corr.loc[f, kept].abs() <= CORR_THRESH).all():
        kept.append(f)

print(f"ANOVA-kept: {len(keep_anova)}  |  After de-correlation: {len(kept)}")

# 5) Final matrix with de-correlated features
X_final = X_sel[kept].copy()

# Save artifacts
anova_tbl.to_csv("anova_table.csv", index=False)
pd.Series(keep_anova, name="anova_keep").to_csv("features_after_anova.csv", index=False)
pd.Series(kept, name="final_features").to_csv("features_after_decorrelation.csv", index=False)

# Optional peek
anova_tbl.head(10)


In [ ]:
# === Random Forest with embedded selection, nested GroupKFold by chromosome ===
import numpy as np, pandas as pd, joblib, os
from statistics import mode
from collections import Counter

from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import average_precision_score, roc_auc_score

from sklearn.ensemble import RandomForestClassifier

# -------------------------------------------------------------------------
# Inputs expected:
#   - X_final: DataFrame with your 153 selected features
#   - y:       Series/array of 0/1 labels aligned to X_final
#   - data:    original DataFrame containing a chromosome column for grouping
# -------------------------------------------------------------------------

# 0) Get chromosome groups
CANDIDATES = ["chr", "chrom", "chromosome", "chrom_id"]
GROUP_COL = next((c for c in CANDIDATES if c in data.columns), None)
if GROUP_COL is None:
    raise ValueError(f"No chromosome column found. Add one of {CANDIDATES} to `data`.")

groups = data.loc[X_final.index, GROUP_COL].astype(str)

# 1) Set outer/inner grouped CV sizes
n_chr = groups.nunique()
OUTER_SPLITS = min(5, max(2, n_chr))    # need at least 2 unique chromosomes
INNER_SPLITS = min(3, max(2, n_chr - 1))

outer_cv = GroupKFold(n_splits=OUTER_SPLITS)
inner_cv = GroupKFold(n_splits=INNER_SPLITS)

print(f"GroupCV by '{GROUP_COL}': outer={OUTER_SPLITS}, inner={INNER_SPLITS}, groups={n_chr} chromosomes")

# 2) Pipeline: log1p -> impute -> embedded RF selector -> RF classifier
log1p_tf = FunctionTransformer(lambda X: np.log1p(np.clip(X, 0, None)), validate=False)

# Base RF; we’ll tune a few keys in the grid
rf_base = RandomForestClassifier(
    n_estimators=600,
    max_depth=None,
    min_samples_leaf=1,
    max_features="sqrt",
    class_weight="balanced",   # handle imbalance
    n_jobs=-1,
    random_state=42
)

pipe = Pipeline([
    ("log1p", log1p_tf),
    ("imp", SimpleImputer(strategy="median")),
    ("sel", SelectFromModel(
        estimator=RandomForestClassifier(
            n_estimators=400,
            max_depth=None,
            min_samples_leaf=1,
            max_features="sqrt",
            class_weight="balanced",
            n_jobs=-1,
            random_state=42
        ),
        threshold=-np.inf,        # rank purely by RF impurity importance
        max_features=None         # tuned below
    )),
    ("clf", rf_base),
])

# 3) Hyperparameter grid (compact but effective)
param_grid = {
    "sel__max_features": [25, 50, 75, 100, None],  # None = keep all 153
    "clf__max_depth": [None, 10, 20],
    "clf__min_samples_leaf": [1, 2, 5],
    "clf__max_features": ["sqrt", "log2", 0.5],    # 0.5 means 50% of features
}

# 4) Nested GroupKFold
fold_ap, fold_roc, chosen_K, selected_masks = [], [], [], []

for i, (tr, te) in enumerate(outer_cv.split(X_final, y, groups), start=1):
    X_tr, X_te = X_final.iloc[tr], X_final.iloc[te]
    y_tr, y_te = y.iloc[tr], y.iloc[te]
    g_tr, g_te = groups.iloc[tr], groups.iloc[te]

    gs = GridSearchCV(
        pipe, param_grid=param_grid, cv=inner_cv,
        scoring="average_precision", n_jobs=-1, verbose=0,
        refit=True, error_score="raise"
    )
    gs.fit(X_tr, y_tr, groups=g_tr)

    best_est = gs.best_estimator_
    y_score = best_est.predict_proba(X_te)[:, 1]

    ap  = average_precision_score(y_te, y_score)
    roc = roc_auc_score(y_te, y_score)
    fold_ap.append(ap); fold_roc.append(roc)
    chosen_K.append(gs.best_params_["sel__max_features"])

    # record selection mask
    mask = best_est.named_steps["sel"].get_support()
    selected_masks.append(mask)

    print(f"[Outer {i}/{OUTER_SPLITS}] K={gs.best_params_['sel__max_features']} | AP={ap:.4f} | ROC={roc:.4f}")

print("\n===== Grouped (by chromosome) Nested CV with RandomForest =====")
print(f"AP  mean±sd: {np.mean(fold_ap):.4f} ± {np.std(fold_ap):.4f}")
print(f"ROC mean±sd: {np.mean(fold_roc):.4f} ± {np.std(fold_roc):.4f}")
print(f"Chosen K per outer fold: {chosen_K}")

# 5) Selection stability across outer folds
sel_freq = np.mean(np.vstack(selected_masks), axis=0)
freq_series = pd.Series(sel_freq, index=X_final.columns).sort_values(ascending=False)
os.makedirs("rf_groupcv_out", exist_ok=True)
freq_series.to_csv("rf_groupcv_out/selection_frequency.csv")
print("Top 15 most frequently selected features:\n", freq_series.head(15))

# 6) Final refit on ALL data using K = mode(chosen_K) (None = keep all)
K_final = mode([k if k is not None else 10**9 for k in chosen_K])
K_final = None if K_final == 10**9 else K_final
print("Refitting final RF pipeline with K =", K_final)

final_pipe = Pipeline([
    ("log1p", log1p_tf),
    ("imp", SimpleImputer(strategy="median")),
    ("sel", SelectFromModel(
        estimator=RandomForestClassifier(
            n_estimators=400, max_depth=None, min_samples_leaf=1,
            max_features="sqrt", class_weight="balanced",
            n_jobs=-1, random_state=42
        ),
        threshold=-np.inf, max_features=K_final
    )),
    ("clf", RandomForestClassifier(
        n_estimators=600, max_depth=None, min_samples_leaf=1,
        max_features="sqrt", class_weight="balanced",
        n_jobs=-1, random_state=42
    )),
])

final_pipe.fit(X_final, y)
joblib.dump(final_pipe, "rf_groupcv_out/rf_groupcv_pipeline.joblib")
print("Saved -> rf_groupcv_out/rf_groupcv_pipeline.joblib")


In [ ]:
# === Random Forest with embedded selection, nested GroupKFold by chromosome ===
import numpy as np, pandas as pd, joblib, os
from statistics import mode
from collections import Counter

from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import average_precision_score, roc_auc_score

from sklearn.ensemble import RandomForestClassifier
# put this at top-level (not inside another function/cell)
def log1p_nonneg(X):
    import numpy as np
    return np.log1p(np.clip(X, 0, None))

from sklearn.preprocessing import FunctionTransformer
log1p_tf = FunctionTransformer(log1p_nonneg, validate=False)
# -------------------------------------------------------------------------
# Inputs expected:
#   - X_final: DataFrame with your 153 selected features
#   - y:       Series/array of 0/1 labels aligned to X_final
#   - data:    original DataFrame containing a chromosome column for grouping
# -------------------------------------------------------------------------

# 0) Get chromosome groups
CANDIDATES = ["chr", "chrom", "chromosome", "chrom_id"]
GROUP_COL = next((c for c in CANDIDATES if c in data.columns), None)
if GROUP_COL is None:
    raise ValueError(f"No chromosome column found. Add one of {CANDIDATES} to `data`.")

groups = data.loc[X_final.index, GROUP_COL].astype(str)

# 1) Set outer/inner grouped CV sizes
n_chr = groups.nunique()
OUTER_SPLITS = min(5, max(2, n_chr))    # need at least 2 unique chromosomes
INNER_SPLITS = min(3, max(2, n_chr - 1))

outer_cv = GroupKFold(n_splits=OUTER_SPLITS)
inner_cv = GroupKFold(n_splits=INNER_SPLITS)

print(f"GroupCV by '{GROUP_COL}': outer={OUTER_SPLITS}, inner={INNER_SPLITS}, groups={n_chr} chromosomes")

# 2) Pipeline: log1p -> impute -> embedded RF selector -> RF classifier
log1p_tf = FunctionTransformer(lambda X: np.log1p(np.clip(X, 0, None)), validate=False)

# Base RF; we’ll tune a few keys in the grid
rf_base = RandomForestClassifier(
    n_estimators=600,
    max_depth=None,
    min_samples_leaf=1,
    max_features="sqrt",
    class_weight="balanced",   # handle imbalance
    n_jobs=-1,
    random_state=42
)

pipe = Pipeline([
    ("log1p", log1p_tf),
    ("imp", SimpleImputer(strategy="median")),
    ("sel", SelectFromModel(
        estimator=RandomForestClassifier(
            n_estimators=400,
            max_depth=None,
            min_samples_leaf=1,
            max_features="sqrt",
            class_weight="balanced",
            n_jobs=-1,
            random_state=42
        ),
        threshold=-np.inf,        # rank purely by RF impurity importance
        max_features=None         # tuned below
    )),
    ("clf", rf_base),
])

# 3) Hyperparameter grid (compact but effective)
param_grid = {
    "sel__max_features": [75],  # None = keep all 153
    #"clf__max_depth": [None, 10, 20],
    #"clf__min_samples_leaf": [1, 2, 5],
    #"clf__max_features": ["sqrt", "log2", 0.5],    # 0.5 means 50% of features
}

# 4) Nested GroupKFold
fold_ap, fold_roc, chosen_K, selected_masks = [], [], [], []

for i, (tr, te) in enumerate(outer_cv.split(X_final, y, groups), start=1):
    X_tr, X_te = X_final.iloc[tr], X_final.iloc[te]
    y_tr, y_te = y.iloc[tr], y.iloc[te]
    g_tr, g_te = groups.iloc[tr], groups.iloc[te]

    gs = GridSearchCV(
        pipe, param_grid=param_grid, cv=inner_cv,
        scoring="average_precision", n_jobs=-1, verbose=0,
        refit=True, error_score="raise"
    )
    gs.fit(X_tr, y_tr, groups=g_tr)

    best_est = gs.best_estimator_
    y_score = best_est.predict_proba(X_te)[:, 1]

    ap  = average_precision_score(y_te, y_score)
    roc = roc_auc_score(y_te, y_score)
    fold_ap.append(ap); fold_roc.append(roc)
    chosen_K.append(gs.best_params_["sel__max_features"])

    # record selection mask
    mask = best_est.named_steps["sel"].get_support()
    selected_masks.append(mask)

    print(f"[Outer {i}/{OUTER_SPLITS}] K={gs.best_params_['sel__max_features']} | AP={ap:.4f} | ROC={roc:.4f}")

print("\n===== Grouped (by chromosome) Nested CV with RandomForest =====")
print(f"AP  mean±sd: {np.mean(fold_ap):.4f} ± {np.std(fold_ap):.4f}")
print(f"ROC mean±sd: {np.mean(fold_roc):.4f} ± {np.std(fold_roc):.4f}")
print(f"Chosen K per outer fold: {chosen_K}")

# 5) Selection stability across outer folds
sel_freq = np.mean(np.vstack(selected_masks), axis=0)
freq_series = pd.Series(sel_freq, index=X_final.columns).sort_values(ascending=False)
os.makedirs("rf_groupcv_out", exist_ok=True)
freq_series.to_csv("rf_groupcv_out/selection_frequency.csv")
print("Top 15 most frequently selected features:\n", freq_series.head(15))



In [ ]:
from sklearn.pipeline import Pipeline
import numpy as np, pandas as pd, joblib, os
from statistics import mode
from collections import Counter

from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import average_precision_score, roc_auc_score

from sklearn.ensemble import RandomForestClassifier
# put this at top-level (not inside another function/cell)
def log1p_nonneg(X):
    import numpy as np
    return np.log1p(np.clip(X, 0, None))

from sklearn.preprocessing import FunctionTransformer
log1p_tf = FunctionTransformer(log1p_nonneg, validate=False)
final_pipe = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("sel", SelectFromModel(
        estimator=RandomForestClassifier(
            n_estimators=400, max_depth=None, min_samples_leaf=1,
            max_features="sqrt", class_weight="balanced",
            n_jobs=-1, random_state=42
        ),
        threshold=-np.inf, max_features=75
    )),
    ("clf", RandomForestClassifier(
        n_estimators=600, max_depth=None, min_samples_leaf=1,
        max_features="sqrt", class_weight="balanced",
        n_jobs=-1, random_state=42
    )),
])


In [ ]:
import numpy as np, pandas as pd, joblib, os
from statistics import mode
from collections import Counter

from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import average_precision_score, roc_auc_score

from sklearn.ensemble import RandomForestClassifier
# put this at top-level (not inside another function/cell)
def log1p_nonneg(X):
    import numpy as np
    return np.log1p(np.clip(X, 0, None))

from sklearn.preprocessing import FunctionTransformer
log1p_tf = FunctionTransformer(log1p_nonneg, validate=False)


final_pipe.fit(X_final, y)
import joblib
joblib.dump(final_pipe, "rf_groupcv_out/rf_groupcv_pipeline.joblib")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay

# --- Split or use held-out test set ---
# If you already have train/test, use those.
# If not, here we show example with a 20% test split:
from sklearn.model_selection import train_test_split
X_tr, X_te, y_tr, y_te = train_test_split(X_final, y, test_size=0.2,
                                          stratify=y, random_state=42) 

# Fit on training only
final_pipe.fit(X_tr, y_tr)

# --- ROC curve ---
y_score = final_pipe.predict_proba(X_te)[:, 1]
fpr, tpr, _ = roc_curve(y_te, y_score)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, color="blue", lw=2, label=f"ROC curve (AUC = {roc_auc:.3f})")
plt.plot([0,1], [0,1], color="gray", lw=1, linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve on Test Set (Gene Start Classifier)")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

# --- Confusion matrix ---
y_pred = final_pipe.predict(X_te)   # default threshold = 0.5
cm = confusion_matrix(y_te, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap="Blues")
plt.title("Confusion Matrix on Test Set (Gene Start Classifier)")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

def plot_feature_importances(pipe, X, top_n=20, out_file=None):
    # Step 1: extract importances as Series
    mask = pipe.named_steps["sel"].get_support()
    selected_features = X.columns[mask]
    importances = pipe.named_steps["clf"].feature_importances_
    fi = pd.Series(importances, index=selected_features).sort_values(ascending=False)

    # Step 2: select top N
    fi_top = fi.head(top_n)

    # Step 3: plot
    plt.figure(figsize=(8, 6))
    sns.barplot(x=fi_top.values, y=fi_top.index, palette="viridis")
    plt.title(f"Top {top_n} Feature Importances (Gene Start Classifier)")
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.tight_layout()
    
    if out_file:
        plt.savefig(out_file, dpi=200)
        print(f"Saved -> {out_file}")
    else:
        plt.show()
    
    return fi

# Example usage:
fi = plot_feature_importances(final_pipe, X_final, top_n=20,
                              out_file="rf_groupcv_out/feature_importances_top20.png")


In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier

# --- groups by chromosome ---
GROUP_COL = next(c for c in ["chr","chrom","chromosome","chrom_id"] if c in data.columns)
groups = data.loc[X_final.index, GROUP_COL].astype(str)

# pick ~20% chromosomes as dev (deterministic)
rng = np.random.RandomState(42)
uniq_chr = np.array(sorted(groups.unique()))
n_dev = max(3, int(np.ceil(0.2 * len(uniq_chr))))
dev_chr = rng.choice(uniq_chr, size=n_dev, replace=False)
is_dev = groups.isin(dev_chr)

def log1p_clip(X):
    X = np.asarray(X); return np.log1p(np.clip(X, 0, None))

pipe = Pipeline([
    ("log1p", FunctionTransformer(log1p_clip, validate=False)),
    ("imp", SimpleImputer(strategy="median")),
    ("sel", SelectFromModel(
        estimator=RandomForestClassifier(
            n_estimators=300, max_depth=None, min_samples_leaf=1,
            max_features="sqrt", class_weight="balanced",
            n_jobs=-1, random_state=42
        ),
        threshold=-np.inf, max_features=None
    )),
    ("clf", RandomForestClassifier(
        n_estimators=300, max_depth=None, min_samples_leaf=1,
        max_features="sqrt", class_weight="balanced",
        n_jobs=-1, random_state=42
    )),
])

param_grid = {"sel__max_features": [25, 50, 75, 100, None]}  # only K
inner = GroupKFold(n_splits=min(3, n_dev))

gs = GridSearchCV(
    pipe, param_grid=param_grid, cv=inner,
    scoring="average_precision", n_jobs=-1, verbose=1, refit=True, error_score="raise"
)
gs.fit(X_final[is_dev], y[is_dev], groups=groups[is_dev])

K_final = gs.best_params_["sel__max_features"]
print("Chosen K on dev chromosomes:", K_final)


In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import make_scorer, recall_score
import matplotlib.pyplot as plt

# --- groups by chromosome ---
GROUP_COL = next(c for c in ["chr","chrom","chromosome","chrom_id"] if c in data.columns)
groups = data.loc[X_final.index, GROUP_COL].astype(str)

# ~20% chromosomes as dev (deterministic)
rng = np.random.RandomState(42)
uniq_chr = np.array(sorted(groups.unique()))
n_dev = max(3, int(np.ceil(0.2 * len(uniq_chr))))
dev_chr = rng.choice(uniq_chr, size=n_dev, replace=False)
is_dev = groups.isin(dev_chr)

# Picklable transform (avoid lambda so parallel CV can pickle the pipeline)
def log1p_clip(X):
    X = np.asarray(X)
    return np.log1p(np.clip(X, 0, None))

pipe = Pipeline([
    ("log1p", FunctionTransformer(log1p_clip, validate=False)),
    ("imp", SimpleImputer(strategy="median")),
    ("sel", SelectFromModel(
        estimator=RandomForestClassifier(
            n_estimators=300, max_depth=None, min_samples_leaf=1,
            max_features="sqrt", class_weight="balanced",
            n_jobs=-1, random_state=42
        ),
        threshold=-np.inf, max_features=None
    )),
    ("clf", RandomForestClassifier(
        n_estimators=300, max_depth=None, min_samples_leaf=1,
        max_features="sqrt", class_weight="balanced",
        n_jobs=-1, random_state=42
    )),
])

# --- We will measure recall at the default 0.5 threshold on the dev CV folds ---
rec_scorer = make_scorer(recall_score, pos_label=1)

param_grid = {"sel__max_features": [25, 50, 75, 100, None]}  # None = keep all
inner = GroupKFold(n_splits=min(3, n_dev))

gs = GridSearchCV(
    pipe, param_grid=param_grid, cv=inner, scoring=rec_scorer,
    n_jobs=-1, verbose=1, refit=True, error_score="raise", return_train_score=False
)
gs.fit(X_final[is_dev], y[is_dev], groups=groups[is_dev])

# ---- Summarize recall for each K ----
cv = gs.cv_results_
tab = pd.DataFrame({
    "K": cv["param_sel__max_features"].data,  # pandas displays None as <NA> sometimes
    "mean_recall": cv["mean_test_score"],
    "std_recall":  cv["std_test_score"],
})
# nicer ordering: 25,50,75,100,None
order = [k for k in [25, 50, 75, 100, None] if (tab["K"] == k).any()]
tab = tab.set_index("K").loc[order].reset_index()

print("Recall by K on dev chromosomes (GroupKFold):")
print(tab.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

# quick plot
plt.figure(figsize=(6,4))
x = [("all" if k is None else str(k)) for k in tab["K"]]
plt.errorbar(x, tab["mean_recall"], yerr=tab["std_recall"], fmt="o-")
plt.xlabel("K (kept features)"); plt.ylabel("Recall (CV mean ± sd)")
plt.title("Recall vs K (dev chromosomes)")
plt.tight_layout(); plt.show()

# Best K by recall (ties use GridSearchCV tie-breaking)
K_final = gs.best_params_["sel__max_features"]
print("Chosen K on dev chromosomes (by recall):", K_final)

# If you also want to see which  features were selected for that K on the dev refit:
best_pipe = gs.best_estimator_
cols_train = list(best_pipe.named_steps["imp"].feature_names_in_)
mask = best_pipe.named_steps["sel"].get_support()
dev_selected_features = pd.Index(cols_train)[mask].tolist()
print(f"Selected {len(dev_selected_features)} features at K={K_final} (dev refit).")


In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import average_precision_score, roc_auc_score

remain_mask = ~is_dev
outer = GroupKFold(n_splits=min(5, groups[remain_mask].nunique()))

# Fix K and use a beefier RF for evaluation
pipe_fixedK = Pipeline([
    ("log1p", FunctionTransformer(log1p_clip, validate=False)),
    ("imp", SimpleImputer(strategy="median")),
    ("sel", SelectFromModel(
        estimator=RandomForestClassifier(
            n_estimators=400, max_depth=None, min_samples_leaf=1,
            max_features="sqrt", class_weight="balanced",
            n_jobs=-1, random_state=42
        ),
        threshold=-np.inf, max_features=K_final
    )),
    ("clf", RandomForestClassifier(
        n_estimators=600, max_depth=None, min_samples_leaf=1,
        max_features="sqrt", class_weight="balanced",
        n_jobs=-1, random_state=42
    )),
])

fold_ap, fold_roc = [], []
for i, (tr, te) in enumerate(outer.split(X_final[remain_mask], y[remain_mask], groups[remain_mask]), start=1):
    X_tr = X_final[remain_mask].iloc[tr]; y_tr = y[remain_mask].iloc[tr]
    X_te = X_final[remain_mask].iloc[te]; y_te = y[remain_mask].iloc[te]
    pipe_fixedK.fit(X_tr, y_tr)
    proba = pipe_fixedK.predict_proba(X_te)[:,1]
    ap  = average_precision_score(y_te, proba)
    roc = roc_auc_score(y_te, proba)
    fold_ap.append(ap); fold_roc.append(roc)
    print(f"[Fold {i}] AP={ap:.4f} ROC={roc:.4f}")

print("CV (remain chromosomes) AP:", np.mean(fold_ap), "±", np.std(fold_ap))
print("CV (remain chromosomes) ROC:", np.mean(fold_roc), "±", np.std(fold_roc))


In [ ]:
import joblib, os
final_pipe = pipe_fixedK.fit(X_final, y)  # reuse same pipeline; fits on all data with K=K_final
os.makedirs("rf_groupcv_out", exist_ok=True)
joblib.dump(final_pipe, "rf_groupcv_out/rf_kfixed_pipeline.joblib")
print("Saved -> rf_groupcv_out/rf_kfixed_pipeline.joblib")


In [ ]:
import numpy as np, pandas as pd
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.ensemble import RandomForestClassifier

# -----------------------------
# Required inputs you already have:
#   X_final : DataFrame of features (rows = windows)
#   y       : Series of binary labels (1=gene start, 0=intergenic), aligned to X_final
#   data    : DataFrame that includes a chromosome column among ["chr","chrom","chromosome","chrom_id"]
# -----------------------------

# --- config ---
N_EXPERIMENTS = 10          # total runs per param set (5 odd→even, 5 even→odd)
SAMPLE_FRAC   = 1.0         # fraction of windows to sample from each side (1.0 = use all)
RANDOM_SEED   = 42          # base seed
POS_LABEL     = 1           # positive class label
STRATIFY      = True        # keep class ratio when sampling

# choose a classifier family and a list of parameter dicts to try
# (feel free to edit/expand param_list)
clf_base = RandomForestClassifier(
    n_estimators=300, max_depth=None, min_samples_leaf=1,
    max_features="sqrt", class_weight="balanced",
    n_jobs=-1, random_state=RANDOM_SEED
)
param_list = [
    {"clf__n_estimators": 300, "clf__max_depth": None},
    {"clf__n_estimators": 600, "clf__max_depth": None},
    {"clf__n_estimators": 600, "clf__max_depth": 20},
]

# ---------------- helpers ----------------
def parse_chr_int(c):
    """Return integer chromosome number if numeric, else None (e.g., X/Y/MT)."""
    s = str(c).lower().replace("chr", "")
    return int(s) if s.isdigit() else None

def sample_index(idx, labels, frac, rng, stratify=True):
    """Sample a fraction of indices, optionally preserving class ratio."""
    if frac >= 0.999:
        return idx
    if stratify:
        i_pos = idx[labels.loc[idx] == POS_LABEL]
        i_neg = idx[labels.loc[idx] != POS_LABEL]
        n_pos = max(1, int(np.floor(len(i_pos) * frac)))
        n_neg = max(1, int(np.floor(len(i_neg) * frac)))
        s_pos = rng.choice(i_pos, size=min(n_pos, len(i_pos)), replace=False) if len(i_pos) else np.array([], dtype=idx.dtype)
        s_neg = rng.choice(i_neg, size=min(n_neg, len(i_neg)), replace=False) if len(i_neg) else np.array([], dtype=idx.dtype)
        return np.concatenate([s_pos, s_neg])
    else:
        n = max(1, int(np.floor(len(idx) * frac)))
        return rng.choice(idx, size=min(n, len(idx)), replace=False)

# ---------------- build odd/even pools ----------------
GROUP_COL = next(c for c in ["chr","chrom","chromosome","chrom_id"] if c in data.columns)
chr_int = data.loc[X_final.index, GROUP_COL].map(parse_chr_int)

odd_idx  = X_final.index[chr_int.apply(lambda v: (v is not None) and (v % 2 == 1))]
even_idx = X_final.index[chr_int.apply(lambda v: (v is not None) and (v % 2 == 0))]

if len(odd_idx) == 0 or len(even_idx) == 0:
    raise ValueError("Odd/even chromosome pools are empty. Check chromosome labels in your data.")

# ---------------- pipeline (normalize → impute → classifier) ----------------
# (Normalization is applied fold-wise by fitting on TRAIN only, as required.)
def make_pipeline(base_clf):
    return Pipeline([
        ("scaler", StandardScaler(with_mean=True, with_std=True)),
        ("imp", SimpleImputer(strategy="median")),
        ("clf", base_clf),
    ])

# ---------------- main loop ----------------
records = []
exp_counter = 0

for params in param_list:
    # make a fresh pipeline for this param set
    pipe = make_pipeline(clone(clf_base)).set_params(**params)

    for i in range(N_EXPERIMENTS):
        rng = np.random.RandomState(RANDOM_SEED + i)

        # two fixed pools
        odd_pool  = np.array(odd_idx)
        even_pool = np.array(even_idx)

        # sample from each pool (optionally stratified)
        odd_sample  = sample_index(odd_pool, y, SAMPLE_FRAC, rng, stratify=STRATIFY)
        even_sample = sample_index(even_pool, y, SAMPLE_FRAC, rng, stratify=STRATIFY)

        # direction: first 5 train on odd, test on even; next 5 reversed
        if i < (N_EXPERIMENTS // 2):
            tr_idx, te_idx = odd_sample, even_sample
            direction = "odd→even"
        else:
            tr_idx, te_idx = even_sample, odd_sample
            direction = "even→odd"

        # fit scaler+imputer on TRAIN only, then evaluate on TEST
        X_tr, X_te = X_final.loc[tr_idx], X_final.loc[te_idx]
        y_tr, y_te = y.loc[tr_idx], y.loc[te_idx]

        fitted = pipe.fit(X_tr, y_tr)
        y_pred = fitted.predict(X_te)

        acc = accuracy_score(y_te, y_pred)
        prec = precision_score(y_te, y_pred, pos_label=POS_LABEL, zero_division=0)
        rec = recall_score(y_te, y_pred, pos_label=POS_LABEL, zero_division=0)

        records.append({
            "experiment": exp_counter,
            "direction": direction,
            **{k.replace("clf__", ""): v for k, v in params.items()},
            "n_train": len(tr_idx),
            "n_test": len(te_idx),
            "accuracy": acc,
            "precision": prec,
            "recall": rec
        })
        exp_counter += 1

# ---------------- results ----------------
results = pd.DataFrame.from_records(records)
# nice ordering
metric_order = ["accuracy","precision","recall"]
param_cols = [c for c in results.columns if c not in ["experiment","direction","n_train","n_test"] + metric_order]
results = results[["experiment","direction"] + param_cols + ["n_train","n_test"] + metric_order]

# per-parameter mean±sd across 10 experiments
summary = (results
           .groupby(param_cols, dropna=False)[metric_order]
           .agg(["mean","std"])
           .sort_values(("recall","mean"), ascending=False))

print("\n=== Per-experiment metrics ===")
print(results.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\n=== Summary by parameter set (mean ± sd over 10 runs) ===")
with pd.option_context("display.float_format", lambda x: f"{x:.4f}"):
    print(summary)

# optional: save
results.to_csv("odd_even_experiments_detailed.csv", index=False)
summary.to_csv("odd_even_experiments_summary.csv")
print("\nSaved: odd_even_experiments_detailed.csv and odd_even_experiments_summary.csv")



In [ ]:
import re
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score

# --- Build features (drop meta, keep only numeric) ---
meta_cols = [c for c in ["gene", "chr", "strand", "TSS_coord", "label"] if c in data.columns]
X_df = (
    data.drop(columns=[c for c in meta_cols if c in data.columns], errors="ignore")
        .select_dtypes(include=["number"])
        .copy()
)
y = data["label"].astype(int)

# Sanity checks
assert "label" not in X_df.columns, "label leaked into X_df!"
assert "TSS_coord" not in X_df.columns, "TSS_coord leaked into X_df!"

print("Total numeric features:", X_df.shape[1])  # should now be 340

# --- Parse Pf3D7 chromosome -> integer 1..14 ---
def pf3d7_chr_to_int(s: str) -> int | None:
    s = str(s).strip()
    # Primary: Pf3D7_01_v3, Pf3D7-01-v3, pf3d7_14_v3, etc.
    m = re.search(r"pf3d7[_\-]?0*([0-9]{1,2})[_\-]?v", s, flags=re.I)
    if m:
        return int(m.group(1))
    # Secondary: plain '01'..'14'
    m2 = re.fullmatch(r"0*([0-9]{1,2})", s)
    if m2:
        return int(m2.group(1))
    # Tertiary fallback: 'chr1', '1', etc.
    s2 = s.lower().replace("chromosome", "").replace("chr", "").strip()
    if s2.isdigit():
        return int(s2)
    # Unknown => None (we'll drop from odd/even splitting)
    return None

if "chr" not in data.columns:
    raise ValueError("No 'chr' column found in `data` for grouping.")
chr_num = data.loc[X_df.index, "chr"].map(pf3d7_chr_to_int)
drop_mask = chr_num.isna()
if drop_mask.any():
    print("Dropping rows with unmapped chromosome labels:", drop_mask.sum())
    # keep only rows with known chromosomes for odd/even split
    X_df = X_df.loc[~drop_mask].copy()
    y = y.loc[~drop_mask].copy()
    chr_num = chr_num.loc[~drop_mask].copy()

# --- Build odd/even index sets ---
odd_idx  = X_df.index[(chr_num % 2) == 1]
even_idx = X_df.index[(chr_num % 2) == 0]
print(f"Odd windows: {len(odd_idx)} | Even windows: {len(even_idx)}")

if len(odd_idx) == 0 or len(even_idx) == 0:
    raise ValueError("Odd/even chromosome pools are empty. Check 'chr' parsing.")

# --- Simple pipeline (log1p + scale + RF); trees don’t need scaling, but it matches your spec ---
def log1p_clip(X): 
    X = np.asarray(X)
    return np.log1p(np.clip(X, 0, None))

pipe = Pipeline([
    ("log1p", FunctionTransformer(log1p_clip, validate=False)),
    ("imp", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler(with_mean=True, with_std=True)),
    ("clf", RandomForestClassifier(
        n_estimators=400, max_depth=None, min_samples_leaf=1,
        max_features="sqrt", class_weight="balanced",
        n_jobs=-1, random_state=42
    )),
])

# --- 10-run odd/even experiment (5 odd→even, 5 even→odd) ---
def run_once(train_idx, test_idx, seed):
    clf = Pipeline(pipe.steps[:-1] + [("clf", RandomForestClassifier(
        n_estimators=400, max_depth=None, min_samples_leaf=1,
        max_features="sqrt", class_weight="balanced",
        n_jobs=-1, random_state=seed
    ))])
    clf.fit(X_df.loc[train_idx], y.loc[train_idx])
    y_hat = clf.predict(X_df.loc[test_idx])
    acc = accuracy_score(y.loc[test_idx], y_hat)
    prec = precision_score(y.loc[test_idx], y_hat, zero_division=0)
    rec = recall_score(y.loc[test_idx], y_hat, zero_division=0)
    return acc, prec, rec

metrics = []
seeds = range(5)  # five runs each direction
# 1) train on odd, test on even
for s in seeds:
    acc, prec, rec = run_once(odd_idx, even_idx, seed=100+s)
    metrics.append(("odd→even", s, acc, prec, rec))
# 2) train on even, test on odd
for s in seeds:
    acc, prec, rec = run_once(even_idx, odd_idx, seed=200+s)
    metrics.append(("even→odd", s, acc, prec, rec))

# --- Report ---
dfm = pd.DataFrame(metrics, columns=["direction","run","accuracy","precision","recall"])
print(dfm)
print("\nSummary by direction:")
print(dfm.groupby("direction")[["accuracy","precision","recall"]].agg(["mean","std"]).round(4))

print("\nOverall (all 10 runs):")
print(dfm[["accuracy","precision","recall"]].agg(["mean","std"]).round(4))

# Optional: save for record
dfm.to_csv("odd_even_10runs_metrics.csv", index=False)


In [ ]:
# ============================================================
# Recall-focused hyperparameter search (odd/even protocol)
# ============================================================
import itertools, joblib, json, matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    recall_score, accuracy_score, precision_score,
    confusion_matrix, roc_curve, auc,
    RocCurveDisplay, ConfusionMatrixDisplay
)
from sklearn.model_selection import train_test_split

# Small, fast grid (expand if you want a deeper search)
param_grid = {
    "n_estimators":      [300, 400],
    "max_depth":         [None, 12],
    "min_samples_leaf":  [1, 2],
    "max_features":      ["sqrt", "log2"],
}
grid_keys   = list(param_grid.keys())
grid_combos = list(itertools.product(*[param_grid[k] for k in grid_keys]))

def make_pipe(**rf_kwargs):
    """Build a fresh pipeline with given RF kwargs."""
    return Pipeline([
        ("log1p", FunctionTransformer(log1p_clip, validate=False)),
        ("imp", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler(with_mean=True, with_std=True)),
        ("clf", RandomForestClassifier(
            class_weight="balanced", n_jobs=-1, random_state=42, **rf_kwargs
        )),
    ])

def eval_params_on_odd_even(params, seeds=range(5)):
    """Average recall across 10 runs (5 seeds each direction)."""
    recalls = []
    for s in seeds:
        # odd → even
        pipe_oe = make_pipe(**params)
        pipe_oe.set_params(clf__random_state=100+s)
        pipe_oe.fit(X_df.loc[odd_idx],  y.loc[odd_idx])
        y_pred = pipe_oe.predict(X_df.loc[even_idx])
        recalls.append(recall_score(y.loc[even_idx], y_pred, zero_division=0))

        # even → odd
        pipe_eo = make_pipe(**params)
        pipe_eo.set_params(clf__random_state=200+s)
        pipe_eo.fit(X_df.loc[even_idx], y.loc[even_idx])
        y_pred = pipe_eo.predict(X_df.loc[odd_idx])
        recalls.append(recall_score(y.loc[odd_idx], y_pred, zero_division=0))

    return float(np.mean(recalls)), float(np.std(recalls))

In [ ]:
# Grid search
from sklearn.pipeline import Pipeline  # <-- add this
grid_rows = []
for combo in grid_combos:
    params = dict(zip(grid_keys, combo))
    mean_rec, std_rec = eval_params_on_odd_even(params)
    row = {**params, "mean_recall": mean_rec, "std_recall": std_rec}
    grid_rows.append(row)
    print(f"Checked {params} -> recall {mean_rec:.4f} ± {std_rec:.4f}")

grid_df = pd.DataFrame(grid_rows).sort_values("mean_recall", ascending=False).reset_index(drop=True)
print("\n=== Recall-focused grid (odd/even; 10 runs) ===")
print(grid_df.head(10))
grid_df.to_csv("rf_recall_grid_oddeven.csv", index=False)

best_params = grid_df.loc[0, grid_keys].to_dict()
print("\nBest params by odd/even recall:", best_params)


In [ ]:
import numpy as np

def coerce_rf_params(d):
    out = dict(d)  # shallow copy
    # cast tree-size params to proper types
    for k in ("n_estimators", "max_depth", "min_samples_leaf"):
        v = out.get(k, None)
        if v is None or v == "None":
            out[k] = None if k == "max_depth" else out.get(k)  # only max_depth may be None in your grid
            continue
        if isinstance(v, (np.integer, int)):
            out[k] = int(v)
        elif isinstance(v, float):
            # if it looks like an int, make it an int
            out[k] = int(round(v))
        else:
            out[k] = int(v)

    # max_features can be "sqrt"/"log2"/int/float; fix integer-like floats
    v = out.get("max_features", None)
    if isinstance(v, float) and np.isfinite(v) and v.is_integer():
        out["max_features"] = int(v)
    return out

best_params = coerce_rf_params(best_params)
print("Cleaned best params:", best_params)



In [ ]:
best_pipe = make_pipe(**best_params)
best_pipe.set_params(clf__random_state=777)
best_pipe.fit(X_tr, y_tr)
y_pred = best_pipe.predict(X_te)
y_prob = best_pipe.predict_proba(X_te)[:, 1]

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_te, y_pred, labels=[0,1])
print("\nConfusion matrix (labels [0,1]):\n", cm)

plt.figure(figsize=(5,4))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0,1]).plot(values_format="d")
plt.title("Random Forest of Gene Start Classifier")
plt.tight_layout()
plt.savefig("Gene_start_confusion_matrix.png", dpi=150)
plt.close()

In [ ]:
# ROC curve
fpr, tpr, thr = roc_curve(y_te, y_prob)
roc_auc = auc(fpr, tpr)
print(f"ROC AUC (holdout): {roc_auc:.4f}")

plt.figure(figsize=(5,4))
RocCurveDisplay(fpr=fpr, tpr=tpr, roc_auc=roc_auc, estimator_name="RF (best)").plot()
plt.title("ROC of Gene Start Classifier (AUC=0.92)")
plt.tight_layout()
plt.savefig("gene_start_roc.png", dpi=300)
plt.close()

In [ ]:
# Also report accuracy/precision/recall on the 70/30 holdout
from sklearn.metrics import accuracy_score, precision_score, recall_score
acc  = accuracy_score(y_te, y_pred)
prec = precision_score(y_te, y_pred, zero_division=0)
rec  = recall_score(y_te, y_pred, zero_division=0)
print(f"Holdout metrics — Acc: {acc:.4f} | Prec: {prec:.4f} | Recall: {rec:.4f}")

# ============================================================
# Save artifacts
# ============================================================
joblib.dump(best_pipe, "rf_recall_best_pipeline.joblib")
with open("rf_recall_best_params.json", "w") as f:
    json.dump(best_params, f, indent=2)

summary = {
    "holdout": {"accuracy": acc, "precision": prec, "recall": rec, "roc_auc": roc_auc},
    "best_params": best_params,
    "grid_top": grid_df.head(10).to_dict(orient="records"),
}
with open("rf_recall_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\nSaved files:")
print(" - rf_recall_grid_oddeven.csv")
print(" - rf_recall_best_params.json")
print(" - rf_recall_summary.json")
print(" - rf_recall_best_pipeline.joblib")
print(" - rf_holdout_confusion_matrix.png")
print(" - rf_holdout_roc.png")